# Mapping Notebook (Pre-GCSI Baseline)
This notebook builds the semantic bridge between cleaned degree/program text and cleaned occupation text.
It is intentionally scoped to mapping candidates and program-level reconstruction before final GCSI scoring.

In [1]:
import pandas as pd
import numpy as np

In [2]:
masters = pd.read_csv("datasets/cleaned/masters_cleaned_for_mapping.csv")
occupations = pd.read_csv("datasets/cleaned/occupations_cleaned_for_mapping.csv")

In [3]:
print("Masters shape:", masters.shape)
print("Occupations shape:", occupations.shape)

print("\nMasters columns:")
print(masters.columns.tolist())

print("\nOccupations columns:")
print(occupations.columns.tolist())

Masters shape: (57085, 6)
Occupations shape: (1120, 3)

Masters columns:
['program_name', 'program_type', 'program_name_clean', 'degree_text', 'degree_norm', 'remove_flag']

Occupations columns:
['OCC_TITLE', 'occupation_text', 'occupation_clean']


In [4]:
masters.head()

,program_name,program_type,program_name_clean,degree_text,degree_norm,remove_flag
0,Economics,MSc,economics,economics,economics,False
1,Political Science and International Affairs,Master,political science and international affairs,political science and international affairs,political science and international affairs,False
2,Business Administration,MBA,business administration,business administration,business administration,False
3,Computer and Information Science,MSc,computer and information science,computer and information science,computer science,False
4,Industrial Engineering and Systems Management,MEng,industrial engineering and systems management,industrial engineering and systems management,industrial engineering and systems management,False


In [5]:
occupations.head()

,OCC_TITLE,occupation_text,occupation_clean
0,Management Occupations,management occupations,management occupations
1,Top Executives,top executives,top executives
2,Chief Executives,chief executives,chief executives
3,General and Operations Managers,general and operations managers,general and operations managers
4,Legislators,legislators,legislators


In [3]:
degree_col = "degree_norm"
occupation_col = "occupation_text"
print("Using degree column:", degree_col)
print("Using occupation column:", occupation_col)

Using degree column: degree_norm
Using occupation column: occupation_text


In [7]:
degree_labels = (
    masters[degree_col]
    .dropna()
    .astype(str)
    .str.strip()
    .drop_duplicates()
    .sort_values()
    .tolist()
)

print("Number of unique degree labels:", len(degree_labels))
degree_labels[:50]

Number of unique degree labels: 23356


["'s accelerator",
 "'s accelerator 1 term",
 "'s foundation business",
 "'s foundation engineering and sciences",
 "'s foundation life science",
 "'s foundation social sciences",
 "'s in arts and cultural management official",
 "'s in business administration and production system official",
 "'s in business management international track",
 "'s in economics",
 "'s in energy",
 "'s in environmental science",
 "'s in finance",
 "'s in industrial engineering and management",
 "'s in international business",
 "'s in international business and management with university of bradford uk",
 "'s in management in virtual environments",
 "'s in marketing and brand management",
 "'s in strategy and management",
 "'s in urbanism regenerating intermediate landscapes official",
 '1 year business administration',
 '15 month gastroenterology',
 '20th and 21st century literature',
 '20th and 21st century music pgdip',
 '21st century literature',
 '21st century media practice',
 '2d fine arts',
 '3conti

In [17]:
# More cleaning bc we still have some weird labels
# inspect worst labels
bad_samples = [x for x in degree_labels if (
    "\\x" in x or
    "official" in x or
    "track" in x or
    "foundation" in x or
    "accelerator" in x or
    "accelerated" in x or
    "online" in x or
    "pgdip" in x or
    "pgd" in x or
    "pgcert" in x or
    "with university" in x or
    "virtual environments" in x or
    "1 year" in x or
    "2 year" in x or
    "15 month" in x or
    "20th" in x or
    x.startswith("'s")
)]

bad_samples[:200]

["'s accelerator",
 "'s accelerator 1 term",
 "'s foundation business",
 "'s foundation engineering and sciences",
 "'s foundation life science",
 "'s foundation social sciences",
 "'s in arts and cultural management official",
 "'s in business administration and production system official",
 "'s in business management international track",
 "'s in economics",
 "'s in energy",
 "'s in environmental science",
 "'s in finance",
 "'s in industrial engineering and management",
 "'s in international business",
 "'s in international business and management with university of bradford uk",
 "'s in management in virtual environments",
 "'s in marketing and brand management",
 "'s in strategy and management",
 "'s in urbanism regenerating intermediate landscapes official",
 '1 year business administration',
 '15 month gastroenterology',
 '20th and 21st century literature',
 '20th and 21st century music pgdip',
 '3d design for virtual environments pgd',
 '3d design for virtual environments pgdip

In [4]:
# stronger cleaning function
import re
import pandas as pd

def clean_degree_for_mapping(text):
    if pd.isna(text):
        return ""

    text = str(text).lower().strip()

    # remove all escaped byte junk and mojibake remnants
    text = re.sub(r"(?:\\x[0-9a-fA-F]{2})+", " ", text)
    text = re.sub(r"[\x80-\x9f]", " ", text)
    text = text.replace("\x92", " ").replace("\u0092", " ")

    # remove broken leading possessive fragments like "'s", "'s in"
    text = re.sub(r"^\s*'?s\s+in\s+", "", text)
    text = re.sub(r"^\s*'?s\s+", "", text)

    # remove explicit duration and format metadata
    text = re.sub(r"\b\d+\s*(year|years|month|months|term|terms|semester|semesters)\b", " ", text)
    text = re.sub(r"\b(one|two|three|four|five|six|seven|eight|nine|ten|eleven|twelve|fifteen|twenty)\s+(year|years|month|months|term|terms|semester|semesters)\b", " ", text)
    text = re.sub(r"\b\d+(st|nd|rd|th)\b", " ", text)

    # strip pathway/marketing/delivery noise
    remove_patterns = [
        r"\bofficial\b",
        r"\binternational track\b",
        r"\btrack\b",
        r"\bonline\b",
        r"\baccelerator\b",
        r"\baccelerated\b",
        r"\bfoundation\b",
        r"\bintermediate\b",
        r"\bpathway\b",
        r"\bstream\b",
        r"\broute\b",
        r"\bentry\b",
        r"\bpostgraduate\b",
        r"\bgraduate\b",
        r"\bfull[\s-]*time\b",
        r"\bpart[\s-]*time\b",
        r"\bwith university of .*?$",
        r"\bpgdip\b",
        r"\bpgd\b",
        r"\bpgcert\b",
        r"\bcoursework\b",
        r"\bthesis and coursework\b",
        r"\bcourse\b",
    ]
    for pattern in remove_patterns:
        text = re.sub(pattern, " ", text)

    # remove generic degree words
    degree_words = [
        r"\bmaster of\b",
        r"\bmasters of\b",
        r"\bmaster\b",
        r"\bmasters\b",
        r"\bmsc\b",
        r"\bms\b",
        r"\bma\b",
        r"\bmba\b",
        r"\bmeng\b",
        r"\bllm\b",
        r"\bmres\b",
        r"\bmphil\b",
        r"\bmfa\b",
        r"\bmed\b",
        r"\bmarch\b",
        r"\bmph\b",
        r"\bprogram\b",
        r"\bprogramme\b",
        r"\bdegree\b",
    ]
    for pattern in degree_words:
        text = re.sub(pattern, " ", text)

    # drop leftover standalone possessive markers and numbers
    text = re.sub(r"\b'?s\b", " ", text)
    text = re.sub(r"\b\d+\b", " ", text)

    # keep letters, digits, and a few separators (for things like 2d/3d, aba/obm)
    text = re.sub(r"[^a-z0-9\s&/+\-]", " ", text)

    # collapse whitespace and separators
    text = re.sub(r"\s+", " ", text).strip()

    return text

In [20]:
# Quick check on known problematic labels
problem_samples = [
    "'s accelerator",
    "'s accelerator 1 term",
    "'s foundation business",
    "'s in business management international track",
    "'s in international business and management with university of bradford uk",
    "20th and 21st century music pgdip",
    "4 1 \\xc2\\x92s in chemistry",
    "\\xc2\\x92s in entrepreneurship",
    "3d design for virtual environments pgdip",
    "accelerated entry online executive",
]

pd.DataFrame({
    "raw": problem_samples,
    "cleaned": [clean_degree_for_mapping(x) for x in problem_samples]
})

,raw,cleaned
0,'s accelerator,
1,'s accelerator 1 term,
2,'s foundation business,business
3,'s in business management international track,business management
4,'s in international business and management wi...,international business and management
5,20th and 21st century music pgdip,and century music
6,4 1 \xc2\x92s in chemistry,in chemistry
7,\xc2\x92s in entrepreneurship,entrepreneurship
8,3d design for virtual environments pgdip,3d design for virtual environments
9,accelerated entry online executive,entry executive


In [11]:
# Apply cleaning function
masters["degree_final"] = masters[degree_col].apply(clean_degree_for_mapping)
masters[["program_name", degree_col, "degree_final"]].sample(30, random_state=42)

,program_name,degree_norm,degree_final
35679,Earth Science,earth science,earth science
5394,Social Science in Service Management,social science in service management,social science in service management
18153,Neurosciences,neurosciences,neurosciences
23971,Cardiovascular Health and Rehabilitation,cardiovascular health and rehabilitation,cardiovascular health and rehabilitation
31114,Environmental Management,environmental management,environmental management
33221,Environment,environment,environment
39380,Germanic Languages and Literatures,germanic languages and literatures,germanic languages and literatures
35322,Engineering and Project Management,engineering and project management,engineering and project management
26142,International Competition Law & Policy,international competition law policy,international competition law policy
29836,International Business Administration (Online),international business administration online,international business administration


In [25]:
# Full-dataset noise profile for degree labels
noise_checks = {
    "escaped_hex": r"\\\\x[0-9a-fA-F]{2}",
    "leading_broken_s": r"^\s*'?s\b",
    "duration_terms": r"\b\d+\s*(year|years|month|months|term|terms|semester|semesters)\b",
    "ordinal_numbers": r"\b\d+(st|nd|rd|th)\b",
    "official": r"\bofficial\b",
    "track": r"\btrack\b",
    "foundation": r"\bfoundation\b",
    "accelerator_or_accelerated": r"\baccelerator\b|\baccelerated\b",
    "online": r"\bonline\b",
    "with_university_of": r"\bwith university of\b",
    "pg_tags": r"\bpgdip\b|\bpgd\b|\bpgcert\b"
}

profile_rows = []
degree_series = masters[degree_col].dropna().astype(str).str.lower().str.strip()

for name, pat in noise_checks.items():
    mask = degree_series.str.contains(pat, regex=True, na=False)
    count = int(mask.sum())
    examples = degree_series[mask].drop_duplicates().head(8).tolist()
    profile_rows.append({
        "pattern": name,
        "count": count,
        "sample_examples": examples,
    })

noise_profile = pd.DataFrame(profile_rows).sort_values("count", ascending=False)
noise_profile

/var/folders/qj/sypsr4hs17z923tv3v1dkt3w0000gn/T/ipykernel_22643/182955671.py:20: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  mask = degree_series.str.contains(pat, regex=True, na=False)


,pattern,count,sample_examples
10,pg_tags,228,"[economics pgdip, electrical and computer engi..."
8,online,119,"[public administration online pathway, leading..."
5,track,113,['s in business management international track...
2,duration_terms,94,"[information technology itm 21 month, informat..."
7,accelerator_or_accelerated,69,[teaching primary and secondary education acce...
1,leading_broken_s,26,['s in business management international track...
3,ordinal_numbers,16,[atmospheric environment 2nd year of physics o...
6,foundation,11,"[accounting foundation stream mcom, education ..."
4,official,9,[international cooperation: sustainable emerge...
9,with_university_of,8,[global shakespeare with university of warwick...


In [13]:
# Diagnose top residual noise terms after current cleaning
import collections

masters["degree_final"] = masters[degree_col].apply(clean_degree_for_mapping)

post_noise_pattern = (
    r"\\\\x[0-9a-fA-F]{2}|\b(?:track|official|foundation|accelerator|accelerated|online|pgdip|pgd|pgcert|pathway|stream|entry|route|graduate|postgraduate)\b"
    r"|^\s*'?s\b|\b\d+\s*(?:year|years|month|months|term|terms|semester|semesters)\b"
)

post_noise = masters[
    masters["degree_final"].str.contains(post_noise_pattern, regex=True, na=False)
][[degree_col, "degree_final"]].drop_duplicates()

tokens = (
    post_noise["degree_final"]
    .str.split()
    .explode()
    .dropna()
    .astype(str)
    .str.strip()
    .loc[lambda s: s.ne("")]
    .tolist()
)

token_counts = collections.Counter(tokens)
top_tokens = pd.DataFrame(token_counts.most_common(40), columns=["token", "count"] )
top_tokens.head(25)

,token,count


In [12]:
# Post-clean audit: what still looks noisy after cleaning?
masters["degree_final"] = masters[degree_col].apply(clean_degree_for_mapping)

post_noise_pattern = (
    r"\\\\x[0-9a-fA-F]{2}|\b(?:track|official|foundation|accelerator|accelerated|online|pgdip|pgd|pgcert|pathway|stream|entry|route|graduate|postgraduate)\b"
    r"|^\s*'?s\b|\b\d+\s*(?:year|years|month|months|term|terms|semester|semesters)\b"
)

post_noise = masters[masters["degree_final"].str.contains(post_noise_pattern, regex=True, na=False)][[degree_col, "degree_final"]].drop_duplicates()

print("Rows still matching noise pattern after cleaning:", len(post_noise))
post_noise.head(50)

Rows still matching noise pattern after cleaning: 0


,degree_norm,degree_final


In [14]:
# Exact remaining noisy rows after cleaning
print("Remaining noisy rows:", len(post_noise))

Remaining noisy rows: 0


In [15]:
# Dataset-level counts after cleaning
masters["degree_final"] = masters[degree_col].apply(clean_degree_for_mapping)

total_rows = len(masters)
non_null_source = masters[degree_col].notna().sum()
non_empty_clean = masters["degree_final"].astype(str).str.strip().ne("").sum()
empty_clean = total_rows - non_empty_clean

print("Total rows in masters dataset:", total_rows)
print("Rows with non-null original degree label:", int(non_null_source))
print("Rows with non-empty cleaned degree label:", int(non_empty_clean))
print("Rows that became empty after cleaning:", int(empty_clean))

Total rows in masters dataset: 57085
Rows with non-null original degree label: 57085
Rows with non-empty cleaned degree label: 57048
Rows that became empty after cleaning: 37


The cleaning hopefully ends here because I used Copilot to do it for me at this point...

In [5]:
occupation_labels = (
    occupations[occupation_col]
    .dropna()
    .astype(str)
    .str.strip()
    .drop_duplicates()
    .sort_values()
    .tolist()
)

print("Number of unique occupation labels:", len(occupation_labels))
occupation_labels[:50]

Number of unique occupation labels: 1120


['accountants and auditors',
 'actors',
 'actors, producers, and directors',
 'actuaries',
 'adhesive bonding machine operators and tenders',
 'administrative law judges, adjudicators, and hearing officers',
 'administrative services managers',
 'adult basic and secondary education and literacy teachers and instructors',
 'advertising and promotions managers',
 'advertising sales agents',
 'advertising, marketing, promotions, public relations, and sales managers',
 'aerospace engineering and operations technicians',
 'aerospace engineers',
 'agents and business managers of artists, performers, and athletes',
 'agricultural and food science technicians',
 'agricultural and food scientists',
 'agricultural engineers',
 'agricultural equipment operators',
 'agricultural inspectors',
 'agricultural sciences teachers, postsecondary',
 'agricultural workers',
 'agricultural workers, all other',
 'air traffic controllers',
 'air traffic controllers and airfield operations specialists',
 'air 

## Semantic Mapping Stage
From this point, we generate threshold-based degree-to-career candidates using Sentence Transformer embeddings.

In [11]:
# Semantic similarity mapping: degree -> occupation candidates (threshold-based, no fixed top-k)
from sentence_transformers import SentenceTransformer, util
import re
import numpy as np

def clean_occupation_for_mapping(text):
    if pd.isna(text):
        return ""
    text = str(text).lower().strip()
    text = re.sub(r"[^a-z0-9\s&/+\-]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

# Ensure degree_final exists
masters["degree_final"] = masters[degree_col].apply(clean_degree_for_mapping)

degree_vocab = (
    masters["degree_final"]
    .dropna()
    .astype(str)
    .str.strip()
    .loc[lambda s: s.ne("")]
    .drop_duplicates()
    .tolist()
)

occupation_vocab = (
    occupations[occupation_col]
    .dropna()
    .astype(str)
    .map(clean_occupation_for_mapping)
    .loc[lambda s: s.ne("")]
    .drop_duplicates()
    .tolist()
)

print("Unique cleaned degrees:", len(degree_vocab))
print("Unique cleaned occupations:", len(occupation_vocab))

model_name = "sentence-transformers/all-MiniLM-L6-v2"
st_model = SentenceTransformer(model_name)

degree_embeddings = st_model.encode(
    degree_vocab,
    batch_size=256,
    show_progress_bar=True,
    normalize_embeddings=True,
 )
occupation_embeddings = st_model.encode(
    occupation_vocab,
    batch_size=256,
    show_progress_bar=True,
    normalize_embeddings=True,
 )

similarity_threshold = 0.30
batch_size = 512
rows = []

for start in range(0, len(degree_vocab), batch_size):
    end = min(start + batch_size, len(degree_vocab))
    sim_matrix = util.cos_sim(degree_embeddings[start:end], occupation_embeddings).cpu().numpy()

    for i, deg in enumerate(degree_vocab[start:end]):
        row_scores = sim_matrix[i]
        valid_idx = np.where(row_scores >= similarity_threshold)[0]

        if valid_idx.size == 0:
            continue

        sorted_idx = valid_idx[np.argsort(row_scores[valid_idx])[::-1]]
        for rank, occ_idx in enumerate(sorted_idx, start=1):
            rows.append({
                "degree_clean": deg,
                "match_rank": rank,
                "career_match": occupation_vocab[int(occ_idx)],
                "similarity": round(float(row_scores[occ_idx]), 4),
            })

degree_career_candidates = pd.DataFrame(rows).sort_values(["degree_clean", "match_rank"])
degree_career_candidates.head(30)

Unique cleaned degrees: 22901
Unique cleaned occupations: 1120


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7743.28it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Batches: 100%|██████████| 5/5 [00:00<00:00, 31.43it/s]


,degree_clean,match_rank,career_match,similarity
812905,2d fine arts,1,fine artists including painters sculptors and ...,0.6129
812906,2d fine arts,2,craft artists,0.5946
812907,2d fine arts,3,multimedia artists and animators,0.5529
812908,2d fine arts,4,art directors,0.5315
812909,2d fine arts,5,art and design workers,0.5297
812910,2d fine arts,6,art drama and music teachers postsecondary,0.5128
812911,2d fine arts,7,arts design entertainment sports and media occ...,0.4686
812912,2d fine arts,8,motion picture projectionists,0.4596
812913,2d fine arts,9,graphic designers,0.4585
812914,2d fine arts,10,makeup artists theatrical and performance,0.4485


In [15]:
# Threshold-based candidate export (no fixed top-5)
print("Similarity threshold:", similarity_threshold)
print("Candidate rows:", len(degree_career_candidates))
print("Degrees with >=1 candidate:", degree_career_candidates["degree_clean"].nunique())

degree_candidate_stats = (
    degree_career_candidates.groupby("degree_clean")
    .agg(
        candidate_count=("career_match", "count"),
        max_similarity=("similarity", "max"),
        mean_similarity=("similarity", "mean"),
    )
    .reset_index()
    .sort_values(["candidate_count", "max_similarity"], ascending=[False, False])
)

print("\nCandidate count summary:")
display(degree_candidate_stats["candidate_count"].describe().to_frame().T)

# Uncomment to save degree-level candidate outputs
# degree_career_candidates.to_csv("datasets/mapping/degree_career_sentence_transformer_candidates_thr30.csv", index=False)
# degree_candidate_stats.to_csv("datasets/mapping/degree_career_sentence_transformer_candidate_stats_thr30.csv", index=False)

degree_career_candidates.head(50)

Similarity threshold: 0.3
Candidate rows: 1215700
Degrees with >=1 candidate: 22641

Candidate count summary:


,count,mean,std,min,25%,50%,75%,max
candidate_count,22641.0,53.694625,56.827321,1.0,15.0,35.0,73.0,568.0


,degree_clean,match_rank,career_match,similarity
812905,2d fine arts,1,fine artists including painters sculptors and ...,0.6129
812906,2d fine arts,2,craft artists,0.5946
812907,2d fine arts,3,multimedia artists and animators,0.5529
812908,2d fine arts,4,art directors,0.5315
812909,2d fine arts,5,art and design workers,0.5297
812910,2d fine arts,6,art drama and music teachers postsecondary,0.5128
812911,2d fine arts,7,arts design entertainment sports and media occ...,0.4686
812912,2d fine arts,8,motion picture projectionists,0.4596
812913,2d fine arts,9,graphic designers,0.4585
812914,2d fine arts,10,makeup artists theatrical and performance,0.4485


In [14]:
# Quick quality audit for threshold-based candidate mapping
print("Unique careers in candidate map:", degree_career_candidates["career_match"].nunique())
print("Top 10 most frequent candidate careers:")
display(degree_career_candidates["career_match"].value_counts().head(10).to_frame("count"))

print("\nSimilarity summary (all candidates):")
display(degree_career_candidates["similarity"].describe().to_frame().T)

print("\nSample finance/accounting degree candidates:")
audit_samples = degree_career_candidates[
    degree_career_candidates["degree_clean"].str.contains("account|finance|bank|audit", regex=True, na=False)
][ ["degree_clean", "match_rank", "career_match", "similarity"] ].head(30)
display(audit_samples)

Unique careers in candidate map: 1119
Top 10 most frequent candidate careers:


,count
career_match,
management analysts,4594
other management occupations,4573
computer and information systems managers,4395
operations research analysts,4366
historians,4332
computer and information research scientists,4305
management occupations,4295
education administrators,4289
general and operations managers,4267



Similarity summary (all candidates):


,count,mean,std,min,25%,50%,75%,max
similarity,1215700.0,0.37602,0.073337,0.3,0.3225,0.3539,0.4065,1.0



Sample finance/accounting degree candidates:


,degree_clean,match_rank,career_match,similarity
288204,accountancy,1,bill and account collectors,0.6282
288205,accountancy,2,bookkeeping accounting and auditing clerks,0.5531
288206,accountancy,3,new accounts clerks,0.5408
288207,accountancy,4,accountants and auditors,0.5221
288208,accountancy,5,credit analysts,0.4964
288209,accountancy,6,financial managers,0.4847
288210,accountancy,7,miscellaneous financial clerks,0.4692
288211,accountancy,8,miscellaneous financial specialists,0.4670
288212,accountancy,9,financial clerks,0.4594
288213,accountancy,10,financial examiners,0.4593


## Program-Level Reconstruction
This section joins threshold-based degree-career candidates back to original program rows (`program_name`, `program_type`).

In [16]:
# Reconstruct candidate mapping back to original program rows (program_name + program_type)
masters["degree_final"] = masters[degree_col].apply(clean_degree_for_mapping)

program_base = masters.copy().reset_index(drop=True)
program_base["program_row_id"] = program_base.index + 1

program_cols = ["program_row_id", "program_name", "program_type", degree_col, "degree_final"]
program_base = program_base[program_cols].rename(columns={degree_col: "degree_source"})

# Long format: one row per program x candidate career
program_career_candidates_long = program_base.merge(
    degree_career_candidates,
    left_on="degree_final",
    right_on="degree_clean",
    how="left",
)

program_career_candidates_long = program_career_candidates_long[[
    "program_row_id",
    "program_name",
    "program_type",
    "degree_source",
    "degree_final",
    "match_rank",
    "career_match",
    "similarity",
]]

# Optional best candidate per program (still threshold-based source)
program_best_candidate = (
    program_career_candidates_long.sort_values(["program_row_id", "similarity"], ascending=[True, False])
    .drop_duplicates(subset=["program_row_id"], keep="first")
    .reset_index(drop=True)
)

print("Program rows:", len(program_base))
print("Program-candidate long rows:", len(program_career_candidates_long))
print("Programs with >=1 candidate:", program_career_candidates_long["program_row_id"].nunique())

# Uncomment to save program-level candidate outputs
# program_career_candidates_long.to_csv("datasets/mapping/program_career_sentence_transformer_candidates_thr30_long.csv", index=False)
# program_best_candidate.to_csv("datasets/mapping/program_career_sentence_transformer_best_thr30.csv", index=False)

program_career_candidates_long.head(20)

Program rows: 57085
Program-candidate long rows: 4357770
Programs with >=1 candidate: 57085


,program_row_id,program_name,program_type,degree_source,degree_final,match_rank,career_match,similarity
0,1,Economics,MSc,economics,economics,1.0,economists,0.8615
1,1,Economics,MSc,economics,economics,2.0,economics teachers postsecondary,0.5289
2,1,Economics,MSc,economics,economics,3.0,sociologists,0.5004
3,1,Economics,MSc,economics,economics,4.0,financial analysts,0.4864
4,1,Economics,MSc,economics,economics,5.0,mathematicians,0.4742
5,1,Economics,MSc,economics,economics,6.0,home economics teachers postsecondary,0.4621
6,1,Economics,MSc,economics,economics,7.0,budget analysts,0.4577
7,1,Economics,MSc,economics,economics,8.0,statisticians,0.4456
8,1,Economics,MSc,economics,economics,9.0,management analysts,0.4232
9,1,Economics,MSc,economics,economics,10.0,financial managers,0.4207
